# Part 1: Neural Network Analysis — Customer Churn

This notebook builds and evaluates a simple feed-forward neural network for a supervised learning problem: predicting customer churn.

Target variable: `churn`  
- `1` = customer churned  
- `0` = customer retained


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, log_loss, confusion_matrix,
    classification_report, ConfusionMatrixDisplay
)

RANDOM_STATE = 42

## Task 1: Dataset Understanding

In [ ]:
df = pd.read_csv("customer_churn_nn.csv")
df.head()

In [ ]:
print("Rows and columns:", df.shape)
print("\nData types:")
print(df.dtypes)

print("\nMissing values by column:")
print(df.isna().sum())

In [ ]:
df.describe(include="all").T

In [ ]:
target_counts = df["churn"].value_counts().sort_index()
target_percent = (target_counts / len(df) * 100).round(2)

print(pd.DataFrame({"count": target_counts, "percent": target_percent}))

ax = target_counts.plot(kind="bar", figsize=(5, 4))
ax.set_title("Target Variable Distribution")
ax.set_xlabel("Churn")
ax.set_ylabel("Count")
ax.set_xticklabels(["Retained (0)", "Churned (1)"], rotation=0)
plt.tight_layout()
plt.savefig("results/target_distribution.png", dpi=200)
plt.show()

### Dataset interpretation

The dataset has **2000 rows** and **17 columns**.

The target variable is `churn`, where `1` indicates that the customer churned and `0` indicates that the customer was retained.

The categorical input features are: `region, plan_type, contract_type, payment_method`.

The numerical input features are: `tenure_months, monthly_charges_inr, avg_login_days_per_month, support_tickets_last_90_days, payment_delay_days, data_usage_gb, satisfaction_score, last_complaint_days_ago, discount_percent, autopay_enabled, referral_count`.

The `customer_id` column is an identifier and is removed before model training because it does not contain predictive behavioral information.


## Task 2: Data Preprocessing

In [ ]:
target_col = "churn"
id_col = "customer_id"

X = df.drop(columns=[target_col, id_col])
y = df[target_col]

categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)

In [ ]:
numeric_transformer = SkPipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = SkPipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numerical_cols),
    ("cat", categorical_transformer, categorical_cols)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

## Task 3: Neural Network Model Building

The model uses:

- Input layer: all preprocessed numerical and encoded categorical features
- Hidden layer: dense neural network layer through `MLPClassifier`
- Activation function: ReLU
- Output layer: binary classification output
- Loss function: binary cross-entropy/log loss
- Optimizer: Adam optimizer

A neural network learns by performing a forward pass, calculating loss, using backpropagation to compute gradients, and updating weights and biases to reduce prediction error.


In [ ]:
baseline_model = MLPClassifier(
    hidden_layer_sizes=(16,),
    activation="relu",
    learning_rate_init=0.001,
    batch_size=32,
    max_iter=100,
    random_state=RANDOM_STATE,
    early_stopping=True,
    n_iter_no_change=10
)

baseline_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", baseline_model)
])

baseline_pipeline.fit(X_train, y_train)

## Task 4: Training and Evaluation

In [ ]:
train_pred = baseline_pipeline.predict(X_train)
test_pred = baseline_pipeline.predict(X_test)

train_proba = baseline_pipeline.predict_proba(X_train)
test_proba = baseline_pipeline.predict_proba(X_test)

print("Training accuracy:", round(accuracy_score(y_train, train_pred), 4))
print("Testing accuracy:", round(accuracy_score(y_test, test_pred), 4))
print("Training loss:", round(log_loss(y_train, train_proba), 4))
print("Testing loss:", round(log_loss(y_test, test_proba), 4))

print("\nClassification report:")
print(classification_report(y_test, test_pred))

In [ ]:
cm = confusion_matrix(y_test, test_pred)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=["Retained (0)", "Churned (1)"]).plot(
    ax=ax, values_format="d", colorbar=False
)
ax.set_title("Confusion Matrix - Baseline Neural Network")
plt.tight_layout()
plt.savefig("results/evaluation_outputs.png", dpi=200)
plt.show()

### Brief interpretation

The model performs well if the testing accuracy is close to the training accuracy and the testing loss is not much higher than the training loss. A confusion matrix helps show whether the model is correctly identifying both retained and churned customers rather than only predicting the majority class.


## Task 5: Hyperparameter Experimentation

In [ ]:
experiments = [
    {
        "name": "Baseline_1_hidden_relu",
        "hidden_layer_sizes": (16,),
        "activation": "relu",
        "learning_rate_init": 0.001,
        "batch_size": 32,
        "max_iter": 100,
    },
    {
        "name": "More_neurons_relu",
        "hidden_layer_sizes": (32,),
        "activation": "relu",
        "learning_rate_init": 0.001,
        "batch_size": 32,
        "max_iter": 100,
    },
    {
        "name": "Two_hidden_layers_relu",
        "hidden_layer_sizes": (32, 16),
        "activation": "relu",
        "learning_rate_init": 0.001,
        "batch_size": 32,
        "max_iter": 100,
    },
    {
        "name": "Lower_lr_tanh",
        "hidden_layer_sizes": (16,),
        "activation": "tanh",
        "learning_rate_init": 0.0005,
        "batch_size": 64,
        "max_iter": 150,
    },
]

results = []
trained_models = {}

for config in experiments:
    model = MLPClassifier(
        hidden_layer_sizes=config["hidden_layer_sizes"],
        activation=config["activation"],
        learning_rate_init=config["learning_rate_init"],
        batch_size=config["batch_size"],
        max_iter=config["max_iter"],
        random_state=RANDOM_STATE,
        early_stopping=True,
        n_iter_no_change=10
    )

    pipeline = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    train_pred = pipeline.predict(X_train)
    test_pred = pipeline.predict(X_test)
    train_proba = pipeline.predict_proba(X_train)
    test_proba = pipeline.predict_proba(X_test)

    results.append({
        "Experiment": config["name"],
        "Hidden Layers": str(config["hidden_layer_sizes"]),
        "Activation": config["activation"],
        "Learning Rate": config["learning_rate_init"],
        "Batch Size": config["batch_size"],
        "Epochs Run": pipeline.named_steps["model"].n_iter_,
        "Train Accuracy": round(accuracy_score(y_train, train_pred), 4),
        "Test Accuracy": round(accuracy_score(y_test, test_pred), 4),
        "Train Loss": round(log_loss(y_train, train_proba), 4),
        "Test Loss": round(log_loss(y_test, test_proba), 4),
    })

    trained_models[config["name"]] = pipeline

comparison_df = pd.DataFrame(results).sort_values("Test Accuracy", ascending=False)
comparison_df

In [ ]:
comparison_df.to_csv("results/model_comparison_table.csv", index=False)

fig, ax = plt.subplots(figsize=(12, 3))
ax.axis("off")
table = ax.table(
    cellText=comparison_df.values,
    colLabels=comparison_df.columns,
    cellLoc="center",
    loc="center"
)
table.auto_set_font_size(False)
table.set_fontsize(7)
table.scale(1, 1.4)
plt.tight_layout()
plt.savefig("results/model_comparison_table.png", dpi=200, bbox_inches="tight")
plt.show()

## Task 6: Final Reflection

### What role do weights and biases play in the model?

Weights control the strength and direction of the relationship between input features and neurons. Biases allow each neuron to shift its activation threshold. During training, the neural network updates weights and biases to reduce prediction error.

### Why is an activation function required?

An activation function introduces non-linearity. Without activation functions, even a multi-layer neural network would behave like a simple linear model. ReLU and tanh help the model learn more complex patterns.

### What happens when the learning rate is too high or too low?

If the learning rate is too high, the model may overshoot the best solution and the loss may become unstable. If the learning rate is too low, training becomes very slow and may stop before reaching a good solution.

### Did the model show signs of underfitting or overfitting?

The model does not show strong signs of overfitting if the training and testing accuracy are close. If training accuracy is high but testing accuracy is much lower, that would suggest overfitting. If both training and testing accuracy are low, that would suggest underfitting. In this project, the best models produced similar training and testing performance, so the fit appears reasonable.
